![Your 6-week roadmap](../../assets/roadmap/week1.png)

# Week 1 Exercise - Solution: Multiple-Choice Questions in the Exam Schema

Great news from Northbridge Institute: Emma liked your extractor so much that she's redesigning next term's midterm around it. 🎉 There's one catch. She wants to add a new **Section D of multiple-choice questions** (MCQs), and our exam schema can't really hold them.

Look again at `gradewise/schemas/exam.py`. `QuestionType` already allows `"mcq"`, but a `Question` has nowhere to put the **options** (A, B, C, D) or the **correct answer**. Today you fix that.

**The rule:** don't edit `gradewise/schemas/`. The rest of GradeWise (the extractor, the database, the week 3 graph) depends on those classes. Instead you'll **extend** them in this notebook, by subclassing, the way you'd add a feature to a library you don't own. If your design is right, everything that works with an `ExamPaper` keeps working with yours.

### 🎯 Your mission
1. **`MCQOption`**: a `label` (`"A"`) and its `text`
2. **`MCQQuestion(Question)`**: adds `options` and `correct_option`, with these rules:
   - an `mcq` question has at least 2 options, unique labels, and a `correct_option` that is one of them
   - any other question type has **no** options and **no** `correct_option`
3. **`MCQExamPaper(ExamPaper)`**: its questions are `MCQQuestion`s, and its `problems()` adds one more check:
   an MCQ must have exactly **one** criterion, worth all its marks (MCQs are all-or-nothing)
4. **`mark_mcq(question, chosen)`**: marks an MCQ answer with plain Python, no LLM needed
5. Extract Emma's draft Section D with `with_structured_output(MCQExamPaper)`, rehearsing for free first

Ten automatic checks at the end of Part 2 tell you how you're doing. They never crash, so run them as often as you like.

💡 **Fast & cheap run.** Two lines in `.env` decide what this notebook costs. `MODEL_MODE=free` runs on OpenRouter's free models for $0 (same `OPENROUTER_API_KEY`, no credits, but only about 50 requests a day), and `DATA_SIZE=small` uses a small slice of the data (2 PDFs · Q1–Q3 · 2 students) when you just want to see the APIs work. Restart the kernel after editing `.env`; the next cell prints what you're running.

In [ ]:
# Imports and environment first, all in one place

import json
import os

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.runnables import RunnableLambda
from pydantic import BaseModel, Field, ValidationError, model_validator

from gradewise import datasets
from gradewise.config import DATA_DIR, settings
from gradewise.exam import print_exam_summary
from gradewise.exam.extract import SYSTEM_PROMPT
from gradewise.models import chat_model
from gradewise.schemas import ExamPaper, Question

load_dotenv(override=True)

# 🔑 cheap and free both use one key, OPENROUTER_API_KEY (from openrouter.ai/keys); custom may need others
for key_name in settings.required_keys:
    api_key = os.getenv(key_name)
    if api_key:
        print(f"✅ {key_name} exists and begins {api_key[:8]}")
    else:
        print(f"❌ {key_name} not set. Please see Part 5 of the setup guide (and remember to SAVE .env!)")
        print("   Using another provider? That's fine: see setup/SWAP_PROVIDER.md")

print(settings.describe_models())
print(datasets.describe())

## Part 1: The material

Here's Emma's draft of Section D: the questions, her marking scheme and the methodology for MCQs.

In [ ]:
# 💡 Emma's draft of next term's new Section D, exactly as it will be printed
SECTION_D_PAPER = """## **Section D: Multiple choice**
Circle ONE answer for each question.

**Question 11**
Which of the following is NOT one of the four necessary conditions for a deadlock? **[1 mark]**
A. Mutual exclusion    B. Hold and wait    C. Starvation    D. Circular wait

**Question 12**
In the TCP/IP model, which layer does TCP belong to? **[1 mark]**
A. Network    B. Transport    C. Data link    D. Application

**Question 13**
Which normal form removes partial dependencies on a composite key? **[1 mark]**
A. 1NF    B. 2NF    C. 3NF    D. BCNF
"""

SECTION_D_SCHEME = """## **Section D: multiple choice [3 marks]**
Question 11: C (1 mark). Question 12: B (1 mark). Question 13: B (1 mark).
Multiple-choice questions are all-or-nothing: no partial credit. If more than one option is circled, award 0.
If an answer is crossed out and another circled, mark the circled one."""

SECTION_D_METHOD = """**Multiple-choice questions.** Award the mark only if exactly one option is chosen and it is
the correct one. Ignore working or notes written beside the options. Never award part of a mark."""

And some test data, shaped the way a model might return it. A few are deliberately broken.

In [ ]:
# 💡 Test data, as a model might return it. Use these to check your classes.
GOOD_MCQ = {
    "id": "Q11",
    "number": "11",
    "question_type": "mcq",
    "max_marks": 1,
    "text": "Which of the following is NOT one of the four necessary conditions for a deadlock?",
    "options": [
        {"label": "A", "text": "Mutual exclusion"},
        {"label": "B", "text": "Hold and wait"},
        {"label": "C", "text": "Starvation"},
        {"label": "D", "text": "Circular wait"},
    ],
    "correct_option": "C",
    "criteria": [{"id": "Q11.c1", "description": "Chooses C (starvation) and only C.", "marks": 1}],
    "methodology": "All-or-nothing. Award the mark only if exactly one option is chosen and it is correct.",
}

WRONG_ANSWER_KEY = {**GOOD_MCQ, "correct_option": "E"}  # ❌ there is no option E
ONE_OPTION = {**GOOD_MCQ, "options": GOOD_MCQ["options"][:1], "correct_option": "A"}  # ❌ not a real choice
DUPLICATE_LABELS = {**GOOD_MCQ, "options": GOOD_MCQ["options"][:2] + [{"label": "A", "text": "Starvation"}]}  # ❌
WRITTEN_WITH_OPTIONS = {**GOOD_MCQ, "question_type": "short"}  # ❌ a written question can't have options
WRITTEN = {k: v for k, v in GOOD_MCQ.items() if k not in ("options", "correct_option")} | {"question_type": "short"}
TWO_CRITERIA = {  # ✅ valid shape, but problems() should complain: MCQs are all-or-nothing
    **GOOD_MCQ,
    "criteria": [
        {"id": "Q11.c1", "description": "Chooses C.", "marks": 0.5},
        {"id": "Q11.c2", "description": "Says why.", "marks": 0.5},
    ],
}


def mini_exam(*questions: dict) -> dict:
    """Wrap questions in a tiny exam whose total matches."""
    return {
        "exam_id": "mcq_demo",
        "title": "Section D",
        "subject": "CS201",
        "questions": list(questions),
        "total_marks": sum(q["max_marks"] for q in questions),
    }


EXPECTED_EXAM = json.loads((DATA_DIR / "exams" / "cs_midterm" / "expected_exam.json").read_text(encoding="utf-8"))

## Part 2: Extend the schema

### 2.1 `MCQOption` and `MCQQuestion`

`MCQQuestion` **inherits** every field of `Question` (`id`, `text`, `criteria`, ...), so you only add what's new. Give `options` and `correct_option` **defaults** (an empty list and `None`): that's what lets a written question, like every question in the CS201 midterm, still validate.

In [ ]:
class MCQOption(BaseModel):
    """One answer option of a multiple-choice question."""

    label: str = Field(description='The option letter as printed, e.g. "A"')
    text: str = Field(description="The option's wording, exactly as printed")


class MCQQuestion(Question):
    """A Question that can also be multiple choice (question_type "mcq")."""

    options: list[MCQOption] = Field(
        default_factory=list, description="For mcq questions only: every option, in printed order. Else empty."
    )
    correct_option: str | None = Field(
        default=None, description='For mcq questions only: the label of the right answer, e.g. "C". Else null.'
    )

    @model_validator(mode="after")
    def _options_match_type(self) -> "MCQQuestion":
        labels = [o.label.strip().upper() for o in self.options]
        if self.question_type != "mcq":
            if self.options or self.correct_option:
                raise ValueError(f"{self.id}: only mcq questions can have options or a correct_option.")
            return self
        if len(labels) < 2:
            raise ValueError(f"{self.id}: an mcq needs at least 2 options, got {len(labels)}.")
        if len(set(labels)) != len(labels):
            raise ValueError(f"{self.id}: option labels must be unique, got {labels}.")
        if (self.correct_option or "").strip().upper() not in labels:
            raise ValueError(f"{self.id}: correct_option {self.correct_option!r} is not one of {labels}.")
        return self

💡 **Why a validator here, but `problems()` for the criteria rule?** Remember Day 3: a validator that raises inside `with_structured_output` throws the model's answer away. These option rules are cheap, unambiguous *shape* rules (an answer key pointing at option E is broken, full stop), so failing fast is fine. The criteria rule is about the *marking*, and a human might reasonably disagree, so it's a `problems()` item the model (or Emma) can fix. 🚀 PRO: if you want `extract_with_retry`-style self-correction for everything, move the option rules into `problems()` too.

### 2.2 `MCQExamPaper`

Override the `questions` field so it holds `MCQQuestion`s, and extend `problems()`: call `super().problems()` to keep every existing check, then add yours.

In [ ]:
class MCQExamPaper(ExamPaper):
    """An ExamPaper whose questions may be multiple choice."""

    questions: list[MCQQuestion] = Field(min_length=1)  # 💡 same field, narrower type

    def problems(self) -> list[str]:
        issues = super().problems()  # 💡 keep every original check: totals, criteria sums, methodology
        for q in self.questions:
            if q.question_type == "mcq" and (len(q.criteria) != 1 or q.criteria[0].marks != q.max_marks):
                issues.append(f"{q.id}: an mcq must have exactly one criterion worth all {q.max_marks:g} marks.")
        return issues

### 2.3 Marking an MCQ without an LLM

Here's a lovely consequence of good structure: once the correct option is *data*, marking an MCQ is one line of Python. No model, no cost, no randomness. The marking scheme's rule "more than one option circled earns 0" belongs here too.

In [ ]:
def mark_mcq(question: MCQQuestion, chosen: str) -> float:
    """Full marks for exactly one correct option, 0 otherwise. Case and spaces don't matter."""
    if question.question_type != "mcq":
        raise ValueError(f"{question.id} is not a multiple-choice question.")
    picked = {part.strip().upper() for part in chosen.replace(",", " ").split()}
    return question.max_marks if picked == {question.correct_option.strip().upper()} else 0.0

### 2.4 The checks

Run this cell whenever you like. Aim for ten green ticks.

In [ ]:
def check(name: str, test) -> None:
    """Run one test and print ✅ or ❌. Never raises, so you can run this cell at any time."""
    try:
        passed = bool(test())
        print(("✅ " if passed else "❌ ") + name)
    except Exception as exc:  # noqa: BLE001  💡 a crash counts as a fail, with the reason
        print(f"❌ {name}  ({type(exc).__name__}: {str(exc).splitlines()[0][:80]})")


def rejects(model, data) -> bool:
    try:
        model.model_validate(data)
    except ValidationError:
        return True
    return False


check("1. a good MCQ validates and keeps its 4 options", lambda: len(MCQQuestion.model_validate(GOOD_MCQ).options) == 4)
check("2. the correct option must be one of the labels", lambda: rejects(MCQQuestion, WRONG_ANSWER_KEY))
check("3. an MCQ needs at least 2 options", lambda: rejects(MCQQuestion, ONE_OPTION))
check("4. option labels must be unique", lambda: rejects(MCQQuestion, DUPLICATE_LABELS))
check("5. a written question can't have options", lambda: rejects(MCQQuestion, WRITTEN_WITH_OPTIONS))
check(
    "6. a written question without options still validates", lambda: MCQQuestion.model_validate(WRITTEN).options == []
)
check(
    "7. the CS201 midterm still loads, with no problems",
    lambda: MCQExamPaper.model_validate(EXPECTED_EXAM).problems() == [],
)
check(
    "8. problems() flags an MCQ that isn't one all-or-nothing criterion",
    lambda: any("Q11" in p for p in MCQExamPaper.model_validate(mini_exam(TWO_CRITERIA)).problems()),
)
check(
    "9. mark_mcq: ' c' earns full marks, 'B' earns 0, 'A, C' earns 0",
    lambda: [mark_mcq(MCQQuestion.model_validate(GOOD_MCQ), a) for a in (" c", "B", "A, C")] == [1, 0, 0],
)
check(
    "10. the JSON schema the model sees includes options and correct_option",
    lambda: {"options", "correct_option"} <= set(json.dumps(MCQExamPaper.model_json_schema()).split('"')),
)

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../../assets/callouts/stop.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ef4444;">Stop & think</h2>
            <span style="color:#ef4444;">Check 7 is the important one: the existing CS201 midterm loads as an <code>MCQExamPaper</code> without a single change. Because <code>MCQExamPaper</code> <i>is an</i> <code>ExamPaper</code>, everything in GradeWise that accepts an <code>ExamPaper</code> (the summary table, the database, the graders) accepts yours too. That's what extending instead of editing buys you.</span>
        </td>
    </tr>
</table>

In [ ]:
midterm = MCQExamPaper.model_validate(EXPECTED_EXAM)
print(isinstance(midterm, ExamPaper), "→ the package's tools work on it unchanged:")
print_exam_summary(midterm)

## Part 3: Extract Section D

Now let the model do the work. We reuse the package's `SYSTEM_PROMPT` (all ten rules still apply) and add an eleventh rule for MCQs:

In [ ]:
MCQ_RULES = """
11. Multiple-choice questions have question_type "mcq". Copy every option into `options`, with its letter as
    `label` and its wording as `text`. Put the letter of the right answer, from the MARKING SCHEME, in
    `correct_option`. Give each MCQ exactly ONE criterion worth all of its marks. Written questions have no
    options and a null correct_option.
"""

mcq_messages = [
    SystemMessage(SYSTEM_PROMPT + MCQ_RULES),  # 💡 the package's checklist, plus one new rule
    HumanMessage(
        'Exam ID: "mcq_demo"\n\n'
        f"=== QUESTION PAPER ===\n{SECTION_D_PAPER}\n\n"
        f"=== MARKING SCHEME ===\n{SECTION_D_SCHEME}\n\n"
        f"=== MARKING METHODOLOGY ===\n{SECTION_D_METHOD}"
    ),
]
print(mcq_messages[1].content[:400], "…")

First a **dress rehearsal** with a pretend model, like on Day 3: it hands back a canned answer, so you can test the plumbing for free.

In [ ]:
class PretendModel:
    """with_structured_output() returns a canned answer, validated against the schema you pass."""

    def __init__(self, answer: dict):
        self.answer = answer

    def with_structured_output(self, schema, **kwargs):
        return RunnableLambda(lambda messages: schema.model_validate(self.answer))


canned = mini_exam(GOOD_MCQ)

rehearsal = PretendModel(canned).with_structured_output(MCQExamPaper).invoke(mcq_messages)
print_exam_summary(rehearsal)
print("Q11 answer key:", rehearsal.questions[0].correct_option)

And the real thing. Section D is short, so this costs well under a cent.

In [ ]:
llm = chat_model(temperature=0)
section_d = llm.with_structured_output(MCQExamPaper).invoke(mcq_messages)
section_d = section_d.model_copy(update={"exam_id": "mcq_demo"})  # ⚠️ we choose the ID, not the model

print_exam_summary(section_d)
for q in section_d.questions:
    options = "  ".join(f"{o.label}. {o.text}" for o in q.options)
    print(f"{q.id}: correct = {q.correct_option} | {options}")

Finally, let's mark Sam's Section D. His circled answers were typed up by Emma (in week 4, GradeWise will read them from his handwriting). Q13 shows the "two options circled" rule in action:

In [ ]:
sam_circled = {"Q11": "C", "Q12": "b", "Q13": "B, C"}

total = 0.0
for q in section_d.questions:
    mark = mark_mcq(q, sam_circled.get(q.id, ""))
    total += mark
    print(f"{q.id}: Sam circled {sam_circled.get(q.id, '-')!r:<8} → {mark:g}/{q.max_marks:g}")
print(f"Section D: {total:g}/{section_d.total_marks:g}, marked for free, in microseconds 🎉")

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../../assets/callouts/business.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#22c55e;">Business value</h2>
            <span style="color:#22c55e;">This is a pattern worth remembering: <b>use the LLM once, to turn documents into structure, then let plain code do the repetitive work</b>. Extracting the answer key costs a fraction of a cent; marking 200 students' MCQs costs nothing and is perfectly consistent. Save the model for what only a model can do: judging written answers.</span>
        </td>
    </tr>
</table>

### 🚀 PRO: a discriminated union

Our `MCQQuestion` uses *optional* fields on one class. An alternative is two classes, a written `Question` and an `MCQQuestion` with *required* options, chosen per item by `question_type`. Pydantic calls this a **discriminated union**:

In [ ]:
from typing import Annotated

from pydantic import Discriminator, Tag


class StrictMCQ(Question):
    options: list[MCQOption] = Field(min_length=2)  # 💡 required here: no defaults
    correct_option: str


def question_kind(value) -> str:
    kind = value.get("question_type") if isinstance(value, dict) else getattr(value, "question_type", None)
    return "mcq" if kind == "mcq" else "written"


AnyQuestion = Annotated[
    Annotated[StrictMCQ, Tag("mcq")] | Annotated[Question, Tag("written")], Discriminator(question_kind)
]


class UnionExamPaper(ExamPaper):
    questions: list[AnyQuestion] = Field(min_length=1)


demo = UnionExamPaper.model_validate(mini_exam(GOOD_MCQ, WRITTEN | {"id": "Q14", "number": "14"}))
print([type(q).__name__ for q in demo.questions])

It's tidier for code that handles each type separately, but the JSON schema the model must fill is bigger and harder (an `anyOf` of two question shapes), and some providers' structured-output modes handle unions less reliably. For one optional feature, the simple subclass is the better trade.

## Recap

You extended GradeWise's exam schema without touching the package: a subclass with defaults kept the old data valid, a validator caught broken answer keys, `problems()` grew one more check via `super()`, and `mark_mcq` showed how structure turns an LLM task into free, deterministic code. 🎉

⏭ **Next:** [Week 2](../../week2/README.md): building GradeWise's knowledge base.

### 🧱 Using the package's retry loop with your schema

`extract_once`, `extract_exam` and `extract_with_retry` in `gradewise/exam/extract.py` take a `schema=` argument, so your `MCQExamPaper` plugs straight into the week 3 self-correcting loop: the model is asked for *your* schema, and *your* `problems()` (including the all-or-nothing MCQ check) drives the retries. Rehearse for free with the pretend model first:

In [ ]:
from gradewise.exam import extract_with_retry

rehearsed = extract_with_retry(
    SECTION_D_PAPER,
    SECTION_D_SCHEME,
    SECTION_D_METHOD,
    exam_id="mcq_demo",
    llm=PretendModel(canned),
    schema=MCQExamPaper,
)
print(
    type(rehearsed).__name__, "→", rehearsed.questions[0].id, "correct option:", rehearsed.questions[0].correct_option
)

And for real. ⚠️ The package's `SYSTEM_PROMPT` doesn't include our rule 11, so the model relies on the field descriptions in `MCQOption` / `MCQQuestion` (another reason those descriptions matter). If it slips, `problems()` sends it back to fix.

In [ ]:
section_d_retry = extract_with_retry(
    SECTION_D_PAPER,
    SECTION_D_SCHEME,
    SECTION_D_METHOD,
    exam_id="mcq_demo",
    llm=chat_model(temperature=0),
    schema=MCQExamPaper,
)
print_exam_summary(section_d_retry)

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../../assets/callouts/gradewise.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#7c5cff;">GradeWise</h2>
            <span style="color:#7c5cff;">The <code>schema=</code> argument is the "extend, don't edit" idea at the package level: GradeWise's extractor works with any <code>ExamPaper</code> subclass. To make MCQs a permanent feature, the remaining step would be a branch in week 5's grader that calls <code>mark_mcq</code> instead of the LLM for <code>mcq</code> questions.</span>
        </td>
    </tr>
</table>